In [1]:
import os
import numpy as np
import subprocess

import spaceKLIP

import matplotlib
matplotlib.rcParams.update({'font.size': 14})
%matplotlib inline

from astropy.io import fits

import webbpsf_ext
webbpsf_ext.setup_logging('WARN', verbose=False)

# download using jwst_download.py or spaceKLIP modules requires MAST API TOKEN 
## - if you have TOKEN prepared, just follow spaceKLIP documentation for downloading
## the below cases show errors when running these modules without API TOKEN

### using jwst_download.py

In [2]:
# Name the root directory where we will keep the data for this tutorial.
data_root = 'data_nircam_hd65426'

# Make subdirectories to put the data in.

os.makedirs(data_root, exist_ok=True)
os.makedirs(os.path.join(data_root, 'uncal'), exist_ok=True)

# Invoke the download.
download_cmd = (
    "yes | jwst_download.py --propID 1386 -i nircam -l 700 --obsnums 1 2 3 "
    "--outsubdir data_nircam_hd65426/uncal --skip_propID2outsubdir "
    "-f uncal --date_select 59789.0+"
)

process=subprocess.Popen(download_cmd, shell=True,
                         stdout=subprocess.PIPE,
                         stderr=subprocess.PIPE)
stdout, stderr = process.communicate()

# Uncomment to print the download log and any errors.
print(stdout.decode())
print(stderr.decode())

INSTRUMENT:  nircam
obsmode:  [None]
propID:  01386
obsnums:  [1, 2, 3]

Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/spaceklip/bin/jwst_download.py", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/anaconda3/envs/spaceklip/lib/python3.11/site-packages/jwst_mast_query/scripts/jwst_download.py", line 23, in main
    download.login(raiseErrorFlag=True)
  File "/usr/local/anaconda3/envs/spaceklip/lib/python3.11/site-packages/jwst_mast_query/jwst_query.py", line 471, in login
    raise RuntimeError("No token!! Cannot login! you can set the token with \$API_MAST_TOKEN, as 'token' in the config file, or with --token ")
RuntimeError: No token!! Cannot login! you can set the token with \$API_MAST_TOKEN, as 'token' in the config file, or with --token 



### using SpaceKLIP.mast.download_files
### now spaceKLIP requires the API TOKEN even if you want to download public data

In [3]:
table = spaceKLIP.mast.query_coron_datasets('NIRCam', program=1386, filt='F444W', kind='SCI', return_filenames=True,
                                    level='uncal')

spaceKLIP.mast.download_files(table)

ValueError: Must define MAST_API_TOKEN env variable or specify mast_api_token parameter

# another way without TOKEN - query data with spaceKLIP.mast but download them with astroquery.mast

## first query data with spaceKLIP

In [4]:
import os
from pathlib import Path

import spaceKLIP
from astroquery.mast import Observations

# Name the root directory where we will keep the data for this tutorial.
data_root = Path("data_nircam_hd65426")
uncal_dir = data_root / "uncal"

# Make subdirectories to put the data in.
uncal_dir.mkdir(parents=True, exist_ok=True)

# Query the same ERS 1386 NIRCam observations 1, 2, 3.
# This replaces:
# jwst_download.py --propID 1386 -i nircam --obsnums 1 2 3 -f uncal ...

table = spaceKLIP.mast.query_coron_datasets(
    "NIRCam",                  # inst:
                               #   Instrument name. Usually "NIRCam" or "MIRI".
                               #   We use "NIRCam" here because this tutorial uses
                               #   JWST/NIRCam coronagraphic data.

    program=1386,              # program:
                               #   JWST Program ID.
                               #   This corresponds to --propID 1386 in jwst_download.py.

    obsnum=[1, 2, 3],           # obsnum:
                               #   Observation Number(s) within the JWST program.
                               #   This corresponds to --obsnums 1 2 3.
                               #   If this is omitted, other coronagraphic observations
                               #   in Program 1386 may also be returned.

    return_filenames=True,     # return_filenames:
                               #   If False, the query returns a higher-level summary table.
                               #   If True, it returns individual product filenames and
                               #   additional metadata.
                               #   This is usually needed before downloading files.

    level="uncal",             # level:
                               #   JWST data product level to retrieve.
                               #   "uncal" means the query should return *_uncal.fits files.
                               #   This corresponds to -f uncal in jwst_download.py.

    ignore_exclusive_access=True,
                               # ignore_exclusive_access:
                               #   If True, restricted / exclusive-access data are excluded.
                               #   This is useful when you want to avoid using a MAST API token
                               #   and download only public data.
)

# Optional: mimic --date_select 59789.0+
# query_coron_datasets returns vststart_mjd, so filter after the query.
table = table[table["vststart_mjd"] >= 59789.0]

# Sanity check.
print(f"Found {len(table)} public uncal files.")
table["filename", "kind", "filter", "coronmsk", "targname", "obslabel", "visit_id", "vststart_mjd", "isRestricted"]

Found 55 public uncal files.


filename,kind,filter,coronmsk,targname,obslabel,visit_id,vststart_mjd,isRestricted
str45,bytes3,str5,str9,str9,str20,str12,float64,bool
jw01386001001_03106_00001_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00002_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00003_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00004_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00005_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00006_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00007_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00008_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False
jw01386001001_03106_00009_nrcalong_uncal.fits,REF,F250M,MASKA335R,* phi Cen,NIRCam 335R - REF,V01386001001,59789.91795701389,False


## Download public files without a MAST API token.

In [5]:
for row in table:
    filename = str(row["filename"])
    uri = f"mast:JWST/product/{filename}"
    local_path = uncal_dir / filename

    if local_path.exists():
        print(f"Already exists: {local_path}")
        continue

    print(f"Downloading {filename}")
    result = Observations.download_file(
        uri,
        local_path=str(local_path),
        cache=False,
    )
    print(result)

('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)


# return to the tutorial

In [6]:
program = 1386  # Define the program.
filt = 'F444W'  # Set to None to disable filter selection and load all filters.

# Initialize spaceKLIP database.

database = spaceKLIP.database.create_database(
                                    input_dir=os.path.join(data_root, 'uncal'),
                                    output_dir=data_root,
                                    filt=filt,
                                    pid=program)

2026-05-27 10:52:57,441 - CRDS - INFO -  Calibration SW Found: jwst 1.17.1 (/usr/local/anaconda3/envs/spaceklip/lib/python3.11/site-packages/jwst-1.17.1.dist-info)


[spaceKLIP.database:INFO] --> Identified 1 concatenation(s)
[spaceKLIP.database:INFO]   --> Concatenation 1: JWST_NIRCAM_NRCALONG_F444W_MASKRND_MASK335R_SUB320A335R
TYPE  EXP_TYPE DATAMODL TELESCOP ...      ROLL_REF      BLURFWHM NANMASKFILE
---- --------- -------- -------- ... ------------------ -------- -----------
 SCI NRC_CORON   STAGE0     JWST ... 110.30193021095504      nan        NONE
 SCI NRC_CORON   STAGE0     JWST ... 120.38229928196697      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03164122132496      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03167106048886      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03169329703867      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03168369754667      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03167513985059      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03166494498083      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03164410548654      nan 

In [7]:
database.summarize()

NIRCAM_F444W_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF


## if you want to reduce all the filter data..

In [8]:
program = 1386  # Define the program.
filt = None  # Set to None to disable filter selection and load all filters.

# Initialize spaceKLIP database.

database = spaceKLIP.database.create_database(
                                    input_dir=os.path.join(data_root, 'uncal'),
                                    output_dir=data_root,
                                    filt=filt,
                                    pid=program)

2026-05-27 10:55:28,601 - CRDS - INFO -  Fetching  /Users/tuyama/crds_cache/references/jwst/nircam/jwst_nircam_psfmask_0229.fits  420.5 K bytes  (1 / 1 files) (0 / 420.5 K bytes)
2026-05-27 10:55:31,035 - CRDS - INFO -  Fetching  /Users/tuyama/crds_cache/references/jwst/nircam/jwst_nircam_psfmask_0218.fits  420.5 K bytes  (1 / 1 files) (0 / 420.5 K bytes)
2026-05-27 10:55:33,300 - CRDS - INFO -  Fetching  /Users/tuyama/crds_cache/references/jwst/nircam/jwst_nircam_psfmask_0230.fits  420.5 K bytes  (1 / 1 files) (0 / 420.5 K bytes)
2026-05-27 10:55:36,385 - CRDS - INFO -  Fetching  /Users/tuyama/crds_cache/references/jwst/nircam/jwst_nircam_psfmask_0258.fits  420.5 K bytes  (1 / 1 files) (0 / 420.5 K bytes)


[spaceKLIP.database:INFO] --> Identified 5 concatenation(s)
[spaceKLIP.database:INFO]   --> Concatenation 1: JWST_NIRCAM_NRCALONG_F250M_MASKRND_MASK335R_SUB320A335R
TYPE  EXP_TYPE DATAMODL TELESCOP ...      ROLL_REF      BLURFWHM NANMASKFILE
---- --------- -------- -------- ... ------------------ -------- -----------
 SCI NRC_CORON   STAGE0     JWST ... 110.30193579627581      nan        NONE
 SCI NRC_CORON   STAGE0     JWST ... 120.38231653827546      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03169182746359      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03166745564518      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03166953518361      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03165934259918      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03165950278486      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03166818602391      nan        NONE
 REF NRC_CORON   STAGE0     JWST ... 110.03165453329406      nan 

In [9]:
database.summarize()

NIRCAM_F250M_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF
NIRCAM_F300M_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF
NIRCAM_F356W_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF
NIRCAM_F410M_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF
NIRCAM_F444W_MASK335R
	STAGE0: 11 files;	2 SCI, 9 REF
